In [1]:
#  Applying the above formula
eps = 0.586
print('Doubling a grey gas absorber would change the absorptivity from {:.3} to {:.3}'.format(eps, 2*eps - eps**2))

Doubling a grey gas absorber would change the absorptivity from 0.586 to 0.829


In [2]:
forcing_rate = 2.6 / 0.02   # W/m2 forcing per fractional increase in epsilon

In [3]:
fractional_increase = (0.829 - 0.586) / 0.586

In [4]:
radiative_forcing = forcing_rate * fractional_increase  # W/m2
print(radiative_forcing)

53.907849829351534


In [5]:
lambda_net = -1.3  #  W/m2/K
ecs = radiative_forcing / -(lambda_net)  # K
print(ecs)

41.467576791808874


In [6]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from numpy import cos, deg2rad, log

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [7]:
#  Open handles to the data files
#  These files are climatologies calculated over the final years of each simulation
datapath = "http://thredds.atmos.albany.edu:8080/thredds/dodsC/CESMA/"        
ctrl = xr.open_dataset(datapath + 'som_1850_f19/clim/som_1850_f19.cam.h0.clim.nc', decode_times=False)
co2 = xr.open_dataset(datapath + 'som_1850_2xCO2/clim/som_1850_2xCO2.cam.h0.clim.nc', decode_times=False)

NameError: name 'xr' is not defined

In [8]:
#  Plot cross-sections of the following anomalies under 2xCO2:
#   - Temperature 
#   - Specific humidity
#   - Relative humidity

fig, axes = plt.subplots(1,3, figsize=(16,6))

ax = axes[0]
CS = ax.contourf(ctrl.lat, ctrl.lev, (co2['T'] - ctrl['T']).mean(dim=('time','lon')), 
                 levels=np.arange(-11,12,1), cmap=plt.cm.seismic)
ax.set_title('Temperature (K)')
fig.colorbar(CS, orientation='horizontal', ax=ax)

ax = axes[1]
CS = ax.contourf(ctrl.lat, ctrl.lev, (co2['Q'] - ctrl['Q']).mean(dim=('time','lon'))*1000,
                 levels=np.arange(-3,3.25,0.25), cmap=plt.cm.seismic)
ax.set_title('Specific humidity (g/kg)')
fig.colorbar(CS, orientation='horizontal', ax=ax)

ax = axes[2]
CS = ax.contourf(ctrl.lat, ctrl.lev, (co2['RELHUM'] - ctrl['RELHUM']).mean(dim=('time','lon')),
                 levels=np.arange(-11,12,1), cmap=plt.cm.seismic)
ax.set_title('Relative humidity (%)')
fig.colorbar(CS, orientation='horizontal', ax=ax)

for ax in axes:
    ax.invert_yaxis()
    ax.set_xticks([-90, -60, -30, 0, 30, 60, 90]);
    ax.set_xlabel('Latitude')
    ax.set_ylabel('Pressure')
    
fig.suptitle('Anomalies for 2xCO2 in CESM slab ocean simulations', fontsize=16);

NameError: name 'plt' is not defined

In [9]:
import climlab
from climlab import constants as const

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [10]:
col1 = climlab.BandRCModel()
print(col1)

NameError: name 'climlab' is not defined

In [11]:
col1.state

NameError: name 'col1' is not defined

In [12]:
col1.q

NameError: name 'col1' is not defined

In [13]:
col1.integrate_years(2)

NameError: name 'col1' is not defined

In [14]:
# Check for energy balance
col1.ASR - col1.OLR

NameError: name 'col1' is not defined

In [15]:
fig, ax = plt.subplots()
ax.plot(col1.Tatm, col1.lev, 'c-', label='default')
ax.plot(col1.Ts, climlab.constants.ps, 'co', markersize=16)
ax.invert_yaxis()
ax.set_xlabel('Temperature (K)', fontsize=16)
ax.set_ylabel('Pressure (hPa)', fontsize=16 )
ax.set_title('Temperature profiles', fontsize = 18)
ax.grid()

NameError: name 'plt' is not defined

In [16]:
col1.absorber_vmr

NameError: name 'col1' is not defined

In [17]:
ozone = xr.open_dataset( datapath + 'som_input/ozone_1.9x2.5_L26_2000clim_c091112.nc')

NameError: name 'xr' is not defined

In [18]:
#  Take global (area-weighted) and annual average
weight_ozone = cos(deg2rad(ozone.lat)) / cos(deg2rad(ozone.lat)).mean(dim='lat')
O3_global = (ozone.O3 * weight_ozone).mean(dim=('lat','lon','time'))
print(O3_global)

NameError: name 'cos' is not defined

In [19]:
fig, ax = plt.subplots()
ax.plot( O3_global*1E6, ozone.lev)
ax.invert_yaxis()
ax.set_xlabel('Ozone (ppm)', fontsize=16)
ax.set_ylabel('Pressure (hPa)', fontsize=16 )
ax.set_title('Global, annual mean ozone concentration', fontsize = 16);

NameError: name 'plt' is not defined

In [20]:
#  Create the column with appropriate vertical coordinate, surface albedo and convective adjustment
col2 = climlab.BandRCModel(lev=ozone.lev)
print( col2)

NameError: name 'climlab' is not defined

In [21]:
#  Set the ozone mixing ratio
col2.absorber_vmr['O3'] = O3_global.values

NameError: name 'O3_global' is not defined

In [22]:
#  Run the model out to equilibrium!
col2.integrate_years(2.)

NameError: name 'col2' is not defined

In [23]:
fig, ax = plt.subplots()
ax.plot( col1.Tatm, np.log(col1.lev/1000), 'c-', label='RCE' )
ax.plot( col1.Ts, 0, 'co', markersize=16 )
ax.plot(col2.Tatm, np.log(col2.lev/1000), 'r-', label='RCE O3' )
ax.plot(col2.Ts, 0, 'ro', markersize=16 )
ax.invert_yaxis()
ax.set_xlabel('Temperature (K)', fontsize=16)
ax.set_ylabel('log(Pressure)', fontsize=16 )
ax.set_title('Temperature profiles', fontsize = 18)
ax.grid(); ax.legend()

NameError: name 'plt' is not defined

In [24]:
col3 = climlab.process_like(col2)
print( col3)

NameError: name 'climlab' is not defined

In [25]:
# Let's double CO2.
col3.absorber_vmr['CO2'] *= 2.

NameError: name 'col3' is not defined

In [26]:
col3.compute_diagnostics()
print( 'The radiative forcing for doubling CO2 is %f W/m2.' % (col2.diagnostics['OLR'] - col3.diagnostics['OLR']))

NameError: name 'col3' is not defined

In [27]:
col3.integrate_years(3)

NameError: name 'col3' is not defined

In [28]:
col3.ASR - col3.OLR

NameError: name 'col3' is not defined

In [29]:
print( 'The Equilibrium Climate Sensitivity is %f K.' % (col3.Ts - col2.Ts))

NameError: name 'col3' is not defined

In [30]:
#  An example with no ozone
col4 = climlab.process_like(col1)
print( col4)

NameError: name 'climlab' is not defined

In [31]:
col4.absorber_vmr['CO2'] *= 2.
col4.compute_diagnostics()
print( 'The radiative forcing for doubling CO2 is %f W/m2.' % (col1.OLR - col4.OLR))

NameError: name 'col4' is not defined

In [32]:
col4.integrate_years(3.)
col4.ASR - col4.OLR

NameError: name 'col4' is not defined

In [33]:
print( 'The Equilibrium Climate Sensitivity is %f K.' % (col4.Ts - col1.Ts))

NameError: name 'col4' is not defined